In [1]:
import torch
print(torch.cuda.is_available())

False


/home/stulcrad/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
print(torch.__version__)

2.13.0+cu130


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Only reasoning models
# del model
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# model_id = "openai/gpt-oss-20b" # Ends reasoning with <|channel|>final<|message|> sequence, ends output message with <|return|>

# model_id = "google/gemma-4-26B-A4B-it" # Ends reasoning with <channel|>, ends output message with <turn|>

model_id = "Qwen/Qwen3.5-27B" # End reasoning with </think>, ends output message with <|im_end|>

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # match Gemma's native dtype, not float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    # quantization_config=bnb_config, 
    device_map="auto",
    torch_dtype="auto",
    # device_map={"": 0}
    )

ImportError: /lib64/libc.so.6: version `GLIBC_2.32' not found (required by /home/stulcrad/.local/lib/python3.12/site-packages/causal_conv1d_cuda.cpython-312-x86_64-linux-gnu.so)

In [28]:
from utils.model_reasoning_utils import extract_harmony_final_channel

message = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi, how are you doing?"},
]

prompt = tokenizer.apply_chat_template(
    message,
    tokenize=True,
    reasoning_effort='low',
    add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(**prompt, max_new_tokens=2000)

KeyboardInterrupt: 

In [ ]:
print(tokenizer.decode(prompt['input_ids'][0], skip_special_tokens=False))
print(tokenizer.decode(output[0][prompt['input_ids'].shape[1]:], skip_special_tokens=False))
# print(extract_harmony_final_channel(response))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-07-23

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

You are a helpful assistant.

<|end|><|start|>user<|message|>Hi, how are you doing?<|end|><|start|>assistant
<|channel|>analysis<|message|>Respond politely.<|end|><|start|>assistant<|channel|>final<|message|>Hello! I'm doing great—thanks for asking. How can I help you today?<|return|>


In [12]:
print(tokenizer.convert_ids_to_tokens(output[0][prompt['input_ids'].shape[1]:].tolist()))

['<|channel|>', 'analysis', '<|message|>', 'Respond', 'Ġpolitely', '.', '<|end|>', '<|start|>', 'assistant', '<|channel|>', 'final', '<|message|>', 'Hello', '!', "ĠI'm", 'Ġdoing', 'Ġgreat', 'âĢĶ', 'thanks', 'Ġfor', 'Ġasking', '.', 'ĠHow', 'Ġcan', 'ĠI', 'Ġhelp', 'Ġyou', 'Ġtoday', '?', '<|return|>']


In [9]:
for token, token_id in zip(tokenizer.convert_ids_to_tokens(output[0][prompt['input_ids'].shape[1]:]), output[0][prompt['input_ids'].shape[1]:]):
    print(f"{token}: {token_id}")

In [ ]:
from utils.TokTrie import TokTrie, build_toktrie_from_tokenizer
from utils.TrieSpanConstrainedProcessorTokenAware import TrieSpanConstrainedProcessorTokenAware
from utils.model_reasoning_utils import reasoning_ended
from utils.system_prompts import SYSTEM_PROMPT_CONSTR_GEN

toktrie = build_toktrie_from_tokenizer(tokenizer)

labels = ["PER", "LOC", "ORG", "MISC"]

# input_text = "Radek was born in Prague. He has a dog named Rex."
input_text = """Radek  

was born in Prague.
"""

reasoning_model=True

gpt_marker = "<|channel|>final<|message|>"
gpt_marker_ids = tokenizer(gpt_marker, add_special_tokens=False).input_ids

qwen_marker = "</think>"
qwen_marker_ids = tokenizer(qwen_marker, add_special_tokens=False).input_ids

processor = TrieSpanConstrainedProcessorTokenAware(
    labels, input_text, tokenizer, toktrie, reasoning_model, reasoning_ended,
    # gpt_marker_ids
    qwen_marker_ids
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT_CONSTR_GEN},
    {"role": "user", "content": input_text},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    reasoning_effort='low',
    add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=True,
).to(model.device)

outputs_constrained = model.generate(
    **prompt,
    max_new_tokens=2000,
    logits_processor=[processor],
)

outputs_unconstrained = model.generate(
    **prompt,
    max_new_tokens=2000,
)

In [25]:
output_ids_constrained = outputs_constrained[0][prompt['input_ids'].shape[1]:]
print("Constrained Generation:")
print(tokenizer.decode(output_ids_constrained, skip_special_tokens=False, clean_up_tokenization_spaces=False))

output_ids_unconstrained = outputs_unconstrained[0][prompt['input_ids'].shape[1]:]
print("\nUnconstrained Generation:")
print(tokenizer.decode(output_ids_unconstrained, skip_special_tokens=False, clean_up_tokenization_spaces=False))

Constrained Generation:
<|channel|>analysis<|message|>Need tag Radek PER, Prague LOC.<|end|><|start|>assistant<|channel|>final<|message|><SPAN><LABEL>PER</LABEL>Radek</SPAN>  

was born in <SPAN><LABEL>LOC</LABEL>Prague</SPAN>.<|return|>

Unconstrained Generation:
<|channel|>analysis<|message|>We need to tag "Radek" as PER, "Prague" as LOC. Output whole text with tags. The text: "Radek  

was born in Prague." Note there are line breaks. We must preserve spacing. So "Radek" at start then line break, then "  " two spaces? Actually there's a blank line "Radek  \n\nwas born in Prague." So we keep. Wrap Radek. Wrap Prague.<|end|><|start|>assistant<|channel|>final<|message|><SPAN><LABEL>PER</LABEL>Radek</SPAN>  

was born in <SPAN><LABEL>LOC</LABEL>Prague</SPAN>.<|return|>
